In [7]:
import os
os.makedirs("/content/drive/MyDrive/restaurant_project", exist_ok=True)

!curl -L -o restaurant_inspections.csv "https://data.cityofnewyork.us/api/views/43nn-pn8j/rows.csv?accessType=DOWNLOAD"

import pandas as pd
df = pd.read_csv("restaurant_inspections.csv")
df_clean = df.drop_duplicates().copy()
df_clean['BORO'] = df_clean['BORO'].replace('0', 'Unknown')
df_clean['INSPECTION DATE'] = pd.to_datetime(df_clean['INSPECTION DATE'], errors='coerce')
df_recent = df_clean[df_clean['INSPECTION DATE'].dt.year >= 2022].copy()

df_recent.to_csv("/content/drive/MyDrive/restaurant_project/restaurant_inspections_processed.csv", index=False)
print("Saved:", df_recent.shape)

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  138M    0  138M    0     0  6070k      0 --:--:--  0:00:23 --:--:-- 7691k


/tmp/ipykernel_4686/1745144003.py:7: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("restaurant_inspections.csv")


Saved: (288602, 27)


In [8]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df_recent = pd.read_csv("/content/drive/MyDrive/restaurant_project/restaurant_inspections_processed.csv")
print(df_recent.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(288602, 27)


In [9]:
!pip install -q chromadb sentence-transformers anthropic

In [12]:
df_recent['INSPECTION DATE'] = pd.to_datetime(df_recent['INSPECTION DATE'], errors='coerce')

establishment_features = df_recent.groupby('CAMIS').agg(
    dba=('DBA', 'first'),
    boro=('BORO', 'first'),
    cuisine=('CUISINE DESCRIPTION', 'first'),
    total_inspections=('CAMIS', 'count'),
    total_critical=('CRITICAL FLAG', lambda x: (x == 'Critical').sum()),
    avg_score=('SCORE', 'mean'),
    latest_inspection=('INSPECTION DATE', 'max'),
    first_inspection=('INSPECTION DATE', 'min')
).reset_index()

establishment_features['critical_rate'] = (
    establishment_features['total_critical'] / establishment_features['total_inspections']
)

temp_keywords = 'Cold TCS|Hot TCS|held above|held at or above|temperature'
df_recent['is_temp_violation'] = df_recent['VIOLATION DESCRIPTION'].str.contains(temp_keywords, case=False, na=False)
temp_counts = df_recent[df_recent['is_temp_violation']].groupby('CAMIS').size().rename('temp_violation_count')
establishment_features = establishment_features.merge(temp_counts, on='CAMIS', how='left')
establishment_features['temp_violation_count'] = establishment_features['temp_violation_count'].fillna(0)

yearly_critical = (
    df_recent[df_recent['CRITICAL FLAG']=='Critical']
    .assign(year=df_recent['INSPECTION DATE'].dt.year)
    .groupby(['CAMIS','year']).size().unstack(fill_value=0)
    .reset_index()
)
for yr in [2024, 2025]:
    if yr not in yearly_critical.columns:
        yearly_critical[yr] = 0

establishment_features = establishment_features.merge(
    yearly_critical[['CAMIS', 2024, 2025]], on='CAMIS', how='left'
).fillna({2024: 0, 2025: 0})
establishment_features['critical_change_24_25'] = establishment_features[2025] - establishment_features[2024]

import os
os.makedirs("/content/drive/MyDrive/restaurant_project", exist_ok=True)
establishment_features.to_csv("/content/drive/MyDrive/restaurant_project/establishment_features.csv", index=False)
print("Saved:", establishment_features.shape)

Saved: (27301, 14)


In [13]:
# Build one text passage per establishment (using your existing feature table + raw violation text)
violation_text = (
    df_recent[df_recent['CRITICAL FLAG']=='Critical']
    .groupby('CAMIS')['VIOLATION DESCRIPTION']
    .apply(lambda x: '; '.join(x.dropna().unique()[:5]))  # cap at 5 unique violations to keep chunks reasonable
    .reset_index()
    .rename(columns={'VIOLATION DESCRIPTION': 'violation_summary'})
)

establishment_features = pd.read_csv("/content/drive/MyDrive/restaurant_project/establishment_features.csv")

rag_docs = establishment_features.merge(violation_text, on='CAMIS', how='left')
rag_docs['violation_summary'] = rag_docs['violation_summary'].fillna('No critical violations recorded')

# Build the actual passage text that gets embedded
rag_docs['passage'] = (
    rag_docs['dba'] + " in " + rag_docs['boro'].astype(str) +
    " (" + rag_docs['cuisine'].astype(str) + "): " +
    "Total inspections: " + rag_docs['total_inspections'].astype(str) +
    ", critical violations: " + rag_docs['total_critical'].astype(str) +
    ", change 2024 to 2025: " + rag_docs['critical_change_24_25'].astype(str) +
    ". Common violations: " + rag_docs['violation_summary']
)

print(rag_docs.shape)
print(rag_docs['passage'].iloc[0])

(27301, 16)
MORRIS PARK BAKE SHOP in Bronx (Bakery Products/Desserts): Total inspections: 11, critical violations: 5, change 2024 to 2025: -1.0. Common violations: Evidence of mice or live mice in establishment's food or non-food areas.; No approved written standard operating procedure for avoiding contamination by refillable returnable containers.; Food, supplies, or equipment not protected from potential source of contamination during storage, preparation, transportation, display, service or from customer’s refillable, reusable container. Condiments not in single-service containers or dispensed directly by the vendor.; Tobacco or electronic cigarette use, eating, or drinking from open container in food preparation, food storage or dishwashing area.


In [14]:
from sentence_transformers import SentenceTransformer
import chromadb

# Load a small, fast, free embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# For prototyping, use a sample first (full 27K is fine too, but sample is faster to test with)
sample = rag_docs.sample(n=5000, random_state=42).reset_index(drop=True)

# Generate embeddings
embeddings = embed_model.encode(sample['passage'].tolist(), show_progress_bar=True)

print("Embeddings shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embeddings shape: (5000, 384)


In [15]:
client = chromadb.Client()
collection = client.create_collection("restaurant_inspections")

collection.add(
    ids=[str(i) for i in sample['CAMIS'].tolist()],
    embeddings=embeddings.tolist(),
    documents=sample['passage'].tolist(),
    metadatas=sample[['dba','boro','cuisine']].to_dict('records')
)

print("Stored:", collection.count(), "documents")

Stored: 5000 documents


In [16]:
query = "restaurants with mice or rodent problems"
query_embedding = embed_model.encode([query])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3
)

for doc in results['documents'][0]:
    print(doc[:200], "\n---")

GOOD TASTE CHINESE RESTAURANT in Manhattan (Chinese): Total inspections: 10, critical violations: 3, change 2024 to 2025: 0.0. Common violations: Evidence of mice or live mice in establishment's food  
---
ORRICO'S ITALIAN RESTAURANT in Bronx (Italian): Total inspections: 12, critical violations: 4, change 2024 to 2025: -2.0. Common violations: Evidence of mice or live mice in establishment's food or no 
---
CASABIANCA PIZZERIA RESTAURANT in Manhattan (Pizza): Total inspections: 12, critical violations: 5, change 2024 to 2025: -1.0. Common violations: Evidence of mice or live mice in establishment's food  
---


In [17]:
!pip install -q transformers accelerate

from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",  # small, free, fast enough for CPU/free-GPU Colab
    device_map="auto"
)

def rag_answer(query, n_results=5):
    query_embedding = embed_model.encode([query])
    results = collection.query(query_embeddings=query_embedding.tolist(), n_results=n_results)
    context = "\n\n".join(results['documents'][0])

    prompt = f"""Answer the question using ONLY the restaurant inspection records below. Cite the specific restaurant name(s) you're basing your answer on. If the records don't contain enough information, say so.

Records:
{context}

Question: {query}

Answer:"""

    output = generator(prompt, max_new_tokens=300, do_sample=False)
    return output[0]['generated_text'][len(prompt):]

answer = rag_answer("Which restaurants have had mice or rodent problems?")
print(answer)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 Based on the inspection records provided, GOOD TASTE CHINESE RESTAURANT in Manhattan has evidence of mice or live mice in its food or non-food areas, while ORRICO'S ITALIAN RESTAURANT in Bronx also has evidence of mice or live mice in its food or non-food areas. Therefore, both GOOD TASTE CHINESE RESTAURANT and ORRICO'S ITALIAN RESTAURANT have had issues related to rodents. Maggie Reilly's is listed as having evidence of mice or live mice in its food or non-food areas but no mention of rodent infestation. Village Taverna has evidence of mice or live mice in its food or non-food areas but no changes made for 2024-2025. Taste Kitchen has evidence of mice or live mice in its food or non-food areas and a critical violation regarding hot TCS food items being held below 140°F. Villages Taverna has evidence of mice or live mice in its food or non-food areas but no changes made for 2024-2025. Therefore, based on the inspection records, GOOD TASTE CHINESE RESTAURANT and ORRICO'S ITALIAN RESTAU

In [19]:
def rag_answer(query, n_results=5):
    query_embedding = embed_model.encode([query])
    results = collection.query(query_embeddings=query_embedding.tolist(), n_results=n_results)
    context = "\n\n".join(results['documents'][0])

    prompt = f"""You must answer using ONLY the restaurants named in the records below. Do not mention any restaurant name that does not appear verbatim in the records below. If you're unsure whether a restaurant was mentioned, do not include it.

Records:
{context}

Question: {query}

Answer, listing only restaurants that appear above:"""

    output = generator(prompt, max_new_tokens=300, do_sample=False)
    answer = output[0]['generated_text'][len(prompt):]
    return answer, context  # return both now

In [21]:
answer, context = rag_answer("Which restaurants have had mice or rodent problems?")
print(answer)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 GOOD TASTE CHINESE RESTAURANT, ORRICO'S ITALIAN RESTAURANT, TASTE KITCHEN, MAGGIE REILLY'S
Based on the information provided:

- **GOOD TASTE CHINESE RESTAURANT** has evidence of mice or live mice in its food or non-food areas.
- **ORRICO'S ITALIAN RESTAURANT** has evidence of mice or live mice in its food or non-food areas.
- **TASTE KITCHEN** has evidence of mice or live mice present in its food and/or non-food areas.
- **MAGGIE REILLY'S** has evidence of mice or live mice in its food or non-food areas. 

Therefore, all four restaurants listed have had issues related to rodents. However, since the question asks for specific restaurants that have had mice or rodent problems, we can list them as follows:

**Good Taste Chinese Restaurant**, **Orrico's Italian Restaurant**, **Taste Kitchen**, **Maggie Reilly's**.


In [22]:
import re

def check_grounding(answer, context):
    answer_names = re.findall(r'[A-Z][A-Za-z\'\.]+(?:\s[A-Z][A-Za-z\'\.]+)*', answer)
    hallucinated = [name for name in answer_names if name not in context]
    return hallucinated

hallucinated = check_grounding(answer, context)
print("Possibly hallucinated:", hallucinated)

Possibly hallucinated: ["MAGGIE REILLY'S\nBased", 'Therefore', 'However', 'Good Taste Chinese Restaurant', "Orrico's Italian Restaurant", 'Taste Kitchen', "Maggie Reilly's"]


In [23]:
def check_grounding(answer, context):
    answer_names = re.findall(r'[A-Z][A-Za-z\'\.]+(?:\s[A-Z][A-Za-z\'\.]+)*', answer)
    # ignore common sentence-starter words, compare case-insensitively
    stopwords = {'Therefore', 'However', 'Based', 'Answer', 'Question'}
    hallucinated = [
        name for name in answer_names
        if name not in stopwords and name.upper() not in context.upper()
    ]
    return hallucinated

hallucinated = check_grounding(answer, context)
print("Possibly hallucinated:", hallucinated)

Possibly hallucinated: ["MAGGIE REILLY'S\nBased"]


In [24]:
for q in [
    "restaurants with temperature control violations",
    "restaurants with the most critical violations",
    "restaurants where violations increased in 2025"
]:
    answer, context = rag_answer(q)
    hallucinated = check_grounding(answer, context)
    print(q)
    print("Hallucinated:", hallucinated)
    print("---")

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


restaurants with temperature control violations
Hallucinated: ['BELAIRE CAFE\nMAX RESTAURANT']
---


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


restaurants with the most critical violations
Hallucinated: ["BENEY'S CHAO KING RESTAURANT", 'KAYLA RESTAURANT\nThe', 'KAYLA RESTAURANT. Both']
---
restaurants where violations increased in 2025
Hallucinated: ['Restaurant\nBased', 'All', 'None']
---
